In [ ]:
import gzip, hashlib, json, os, shutil, subprocess, sys
from pathlib import Path
import tensorflow as tf

RUN_ID = "local"
os.environ["DOODLEAI_RUN_ID"] = RUN_ID
os.environ["PYTHONUNBUFFERED"] = "1"
WORK = Path("/kaggle/temp/doodleai-hidden")
RUNS = Path("/kaggle/working/runs")
DATA = Path("/kaggle/working/hidden-data")
SRC = Path("/kaggle/temp/doodleai-src")
print("Run:", RUN_ID)
print("GPUs:", tf.config.list_physical_devices("GPU"))
assert tf.config.list_physical_devices("GPU"), "No GPU: turn one on before training."


In [ ]:
subprocess.run(["git", "clone", "https://github.com/alanszuster/doodleai", str(SRC)], check=True)
subprocess.run(["git", "-C", str(SRC), "fetch", "origin", "main"], check=True)
subprocess.run(["git", "-C", str(SRC), "checkout", "--detach", "-q", "refs/remotes/origin/main"], check=True)
os.chdir(SRC)
subprocess.run(["git", "log", "--oneline", "-1"], check=True)


In [ ]:
subprocess.run([sys.executable, "model/training/prepare_hidden.py", "--work-dir", str(WORK), "--samples", "25209", "--negative-samples", "60000", "--baseline-samples", "24000", "--workers", "4"], check=True)
expected = {
    "X_val.npy": "b6506c8e07d7aa2ab1058d76371ab552e446cfbdf151e357b2462af6643d835f",
    "y_val.npy": "59d80e175444be3828e86516c8f56818504ae1c9ec20df14c3ed638d6d2b21d6",
    "X_test.npy": "c9cedcc6687b98c15d4b6b9fb1a0243ec1a58f92a6d145dbe7a17f2c67f7dba0",
    "y_test.npy": "59d80e175444be3828e86516c8f56818504ae1c9ec20df14c3ed638d6d2b21d6",
}
for name, digest in expected.items():
    if hashlib.sha256((WORK / "data" / name).read_bytes()).hexdigest() != digest:
        raise ValueError(f"Baseline split changed: {name}")
with gzip.open(WORK / "data" / "test_ids.json.gz", "rt") as handle:
    ids = json.load(handle)
if hashlib.sha256(json.dumps(ids, separators=(",", ":")).encode()).hexdigest() != "7acc072185c9d0d40f1ae91a1bc0b81444689757eddc79081299100f9c3f2fcd":
    raise ValueError("Baseline test drawing IDs changed")
DATA.mkdir(parents=True, exist_ok=True)
for name in ("X_val.npy", "y_val.npy", "X_test.npy", "y_test.npy", "meta.json", "test_ids.json.gz"):
    shutil.copy2(WORK / "data" / name, DATA / name)


In [ ]:
subprocess.run([sys.executable, "model/training/train.py", "--work-dir", str(WORK), "--runs-dir", str(RUNS), "--name", "hidden", "--epochs", "20"], check=True)


In [ ]:
subprocess.run([sys.executable, "model/training/evaluate.py", "--model", str(RUNS / "hidden" / "best.keras"), "--data-dir", str(WORK / "data"), "--out", str(RUNS / "hidden" / "report")], check=True)
